# 6.6 — Loss Functions

A loss function turns a model's mistake into one scalar objective, so optimization knows which direction is "less wrong." In this lesson, we build common losses from scratch in NumPy, inspect their shapes and gradients, and connect local arithmetic — squared error, softmax cross-entropy, margins, and embedding distances — to the training behavior they create.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build loss functions one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is exposed so the loss is not a black box. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, vectorized arithmetic, and small numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any random examples.

### 1. A loss is a scalar score for wrongness

Training needs one number to minimize. A model can output many predictions, but the optimizer needs a scalar loss that says how costly those predictions are against the target. For regression, the simplest building block is the residual $e_i=\hat y_i-y_i$: positive means the model overshot, negative means it undershot, and zero means perfect.

In [ ]:
y_w = np.array([2.0, -1.0, 3.0])        # true regression targets.
yhat_w = np.array([1.5, -0.5, 4.0])     # model predictions.
resid_w = yhat_w - y_w                  # signed errors: prediction minus target.
print("targets:", y_w)
print("predictions:", yhat_w)
print("residuals:", resid_w)
assert np.allclose(resid_w, [-0.5, 0.5, 1.0])

▶ What you'll see: three signed mistakes; the third prediction is one full unit too high.

In [ ]:
abs_loss_w = np.mean(np.abs(resid_w))       # average absolute wrongness.
sq_loss_w = np.mean(resid_w ** 2)           # average squared wrongness.
print("mean absolute error:", round(abs_loss_w, 3))
print("mean squared error:", round(sq_loss_w, 3))
assert round(abs_loss_w, 3) == 0.667
assert round(sq_loss_w, 3) == 0.500

▶ What you'll see: the same residuals become two different scalar losses.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["e0", "e1", "e2"], resid_w, color=["steelblue", "steelblue", "crimson"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("1: residuals before choosing a loss")
plt.ylabel("prediction - target")
plt.show()

▶ What you'll see: signed residual bars show direction; a loss decides how much each bar should matter.

*Why it's done this way:* the optimizer cannot minimize a vector of complaints directly; reducing many residuals to one scalar defines the training objective. Different reductions encode different values: absolute error treats every unit equally, while squared error makes large residuals disproportionately important.

### 2. Mean squared error makes large misses loud

Mean squared error is $$L_{MSE}=\frac{1}{n}\sum_i(\hat y_i-y_i)^2.$$ Squaring does three useful things: it removes the sign, makes the loss smooth, and punishes large errors more than small ones. The gradient with respect to a prediction is $\frac{2}{n}(\hat y_i-y_i)$, so each prediction is pushed back toward its target in proportion to its residual.

In [ ]:
grid_w = np.linspace(-3, 3, 61)       # possible residual values.
mse_curve_w = grid_w ** 2             # squared penalty for each residual.
mae_curve_w = np.abs(grid_w)          # absolute penalty for comparison.
print("loss at residual 1:", 1 ** 2)
print("loss at residual 2:", 2 ** 2)
assert 2 ** 2 == 4

▶ What you'll see: doubling the residual from 1 to 2 quadruples the squared penalty.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.plot(grid_w, mse_curve_w, label="squared error", color="purple")
plt.plot(grid_w, mae_curve_w, label="absolute error", color="gray", linestyle="--")
plt.xlabel("residual e")
plt.ylabel("penalty")
plt.title("2: squared loss grows faster")
plt.legend()
plt.show()

▶ What you'll see: the squared-error bowl is gentle near 0 and steep for large misses.

In [ ]:
grad_mse_w = 2 * resid_w / len(resid_w)      # gradient of mean squared error wrt predictions.
print("MSE gradient wrt yhat:", np.round(grad_mse_w, 3))
assert np.allclose(np.round(grad_mse_w, 3), [-0.333, 0.333, 0.667])

▶ What you'll see: the largest positive residual receives the largest positive gradient, so gradient descent will lower that prediction most.

*Why it's done this way:* the parabola gives a clear descent direction everywhere and weights the gradient by error size. That is excellent when big misses really are much worse, but it also means outliers can dominate training.

### 3. Cross-entropy punishes confident wrong probabilities

Classification losses compare probability assigned to the true class. For one-hot target $y$ and predicted probabilities $p$, cross-entropy is $$L_{CE}=-\sum_i y_i\log p_i.$$ Because one entry of $y$ is 1, this reduces to $-\log(p_{true})$: high true-class probability gives low loss, while tiny true-class probability explodes.

In [ ]:
p_good_w = np.array([0.05, 0.90, 0.05])       # confident on class 1.
p_bad_w = np.array([0.80, 0.10, 0.10])        # confident on the wrong class.
y_class_w = 1                                  # true class index.
loss_good_w = -np.log(p_good_w[y_class_w])
loss_bad_w = -np.log(p_bad_w[y_class_w])
print("good CE:", round(loss_good_w, 3))
print("bad CE:", round(loss_bad_w, 3))
assert round(loss_good_w, 3) == 0.105
assert round(loss_bad_w, 3) == 2.303

▶ What you'll see: giving the true class 0.90 probability is cheap; giving it 0.10 probability is much worse.

In [ ]:
probs_w = np.linspace(0.01, 1.0, 100)       # possible probabilities for the true class.
ce_curve_w = -np.log(probs_w)               # cross-entropy for the true class.
plt.figure(figsize=(4.6, 3))
plt.plot(probs_w, ce_curve_w, color="crimson")
plt.xlabel("probability on true class")
plt.ylabel("-log(p_true)")
plt.title("3: cross-entropy loss curve")
plt.show()

▶ What you'll see: the curve is near 0 at probability 1 and rises sharply near probability 0.

In [ ]:
ratio_w = loss_bad_w / loss_good_w
print("bad/good loss ratio:", round(ratio_w, 1))
assert round(ratio_w, 1) == 21.9

▶ What you'll see: the confident wrong distribution is about 22× more costly than the confident right one.

*Why it's done this way:* classification cares about calibrated belief, not just the winning label. The logarithm makes being confidently wrong very expensive, which creates a strong corrective gradient when the model assigns too little probability to the true class.

### 4. Softmax turns scores into probabilities before cross-entropy

Models usually output logits, not probabilities. Softmax converts scores $z$ into probabilities $$p_i=\frac{e^{z_i}}{\sum_j e^{z_j}}.$$ The loss then compares those probabilities with the target. Subtracting the maximum logit before exponentiating is a numerical-stability trick: it changes neither the probabilities nor the loss, but prevents overflow.

In [ ]:
logits_w = np.array([2.15, 0.40, -0.20])      # raw class scores from a model.
shifted_w = logits_w - np.max(logits_w)       # stable logits with max 0.
exp_w = np.exp(shifted_w)                     # exponentiated shifted scores.
probs_soft_w = exp_w / np.sum(exp_w)          # softmax probabilities.
print("shifted logits:", np.round(shifted_w, 3))
print("softmax probabilities:", np.round(probs_soft_w, 3))
assert np.allclose(np.round(probs_soft_w, 3), [0.788, 0.137, 0.075])

▶ What you'll see: class 0 gets most probability because its logit is largest.

In [ ]:
probs_soft_w = np.exp(logits_w - np.max(logits_w)) / np.sum(np.exp(logits_w - np.max(logits_w)))
ce_soft_w = -np.log(probs_soft_w[0])
print("p(true class 0):", round(probs_soft_w[0], 3))
print("softmax CE:", round(ce_soft_w, 3))
assert round(probs_soft_w[0], 3) == 0.788
assert round(ce_soft_w, 3) == 0.238

▶ What you'll see: the lesson-style score 2.15 becomes a probability around 0.788 once all three classes compete.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["class 0", "class 1", "class 2"], probs_soft_w, color="teal")
plt.title("4: softmax probabilities from logits")
plt.ylabel("probability")
plt.ylim(0, 1)
plt.show()

▶ What you'll see: probabilities sum to 1, so raising one class necessarily lowers the others.

*Why it's done this way:* logits are unconstrained and easy for neural networks to produce; probabilities must be nonnegative and sum to 1. Softmax provides that bridge, and the max-shift keeps the same ratios while making the exponentials safe for hardware.

### 5. The cross-entropy gradient is prediction minus target

For softmax plus cross-entropy, a remarkable simplification occurs: the gradient with respect to logits is $p-y$. The model raises logits where predicted probability is below the target and lowers logits where it assigned probability to incorrect classes. This is why CE is such a clean classification training signal.

In [ ]:
y_onehot_w = np.array([1.0, 0.0, 0.0])          # class 0 is the true label.
grad_logits_w = probs_soft_w - y_onehot_w       # softmax CE gradient wrt logits.
print("probabilities:", np.round(probs_soft_w, 3))
print("gradient p-y:", np.round(grad_logits_w, 3))
assert round(float(np.sum(grad_logits_w)), 6) == 0.0

▶ What you'll see: the true class has a negative gradient, while incorrect classes have positive gradients.

In [ ]:
eta_w = 0.6                                      # learning rate for one illustrative logit step.
new_logits_w = logits_w - eta_w * grad_logits_w  # gradient descent on logits.
new_probs_w = np.exp(new_logits_w - np.max(new_logits_w)) / np.sum(np.exp(new_logits_w - np.max(new_logits_w)))
print("old p_true:", round(probs_soft_w[0], 3))
print("new p_true:", round(new_probs_w[0], 3))
assert new_probs_w[0] > probs_soft_w[0]

▶ What you'll see: one descent step raises the probability of the true class.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["old p0", "new p0"], [probs_soft_w[0], new_probs_w[0]], color=["gray", "seagreen"])
plt.ylim(0, 1)
plt.title("5: CE gradient raises the true class")
plt.ylabel("probability")
plt.show()

▶ What you'll see: the true-class probability bar increases after moving opposite the gradient.

*Why it's done this way:* $p-y$ directly measures probability mismatch. Gradient descent subtracts it, so the true class logit rises when $p_{true}<1$, and wrong-class logits fall in proportion to the probability they stole.

### 6. Hinge loss cares about margins, not probabilities

Hinge loss is common in margin-based classifiers. For label $y\in\{-1,+1\}$ and score $s$, $$L=\max(0,1-y s).$$ It is zero only when the correct class score has margin at least 1. Unlike cross-entropy, it does not keep rewarding ever-larger confidence once the margin is satisfied.

In [ ]:
scores_w = np.array([-1.0, 0.2, 0.8, 1.5])       # classifier scores for positive examples.
labels_w = np.ones_like(scores_w)                # all examples have y=+1.
hinge_w = np.maximum(0.0, 1 - labels_w * scores_w)
print("scores:", scores_w)
print("hinge losses:", hinge_w)
assert np.allclose(hinge_w, [2.0, 0.8, 0.2, 0.0])

▶ What you'll see: scores below the margin pay loss; a score of 1.5 pays zero.

In [ ]:
score_grid_w = np.linspace(-2, 3, 100)
hinge_curve_w = np.maximum(0.0, 1 - score_grid_w)
plt.figure(figsize=(4.6, 3))
plt.plot(score_grid_w, hinge_curve_w, color="darkorange")
plt.axvline(1, color="black", linestyle="--", label="margin = 1")
plt.xlabel("y × score")
plt.ylabel("hinge loss")
plt.title("6: margin loss")
plt.legend()
plt.show()

▶ What you'll see: the loss is a straight line until the margin, then becomes exactly flat.

In [ ]:
grad_hinge_w = np.where(labels_w * scores_w < 1, -labels_w, 0.0)
print("hinge gradients wrt score:", grad_hinge_w)
assert np.allclose(grad_hinge_w, [-1.0, -1.0, -1.0, 0.0])

▶ What you'll see: every margin violator receives the same push; the already-safe example receives none.

*Why it's done this way:* hinge loss encodes a different notion of success: not calibrated probability, but enough separation. Once the margin is achieved, extra confidence is irrelevant to this objective, which can make training focus on boundary cases.

### 7. Contrastive and triplet losses shape distances

Representation learning often trains embeddings rather than direct labels. A contrastive loss pulls matching pairs together and pushes nonmatching pairs apart; a triplet loss compares an anchor, a positive, and a negative so the negative is at least a margin farther away than the positive.

In [ ]:
anchor_w = np.array([1.0, 1.0])
positive_w = np.array([1.2, 0.9])
negative_w = np.array([2.2, 1.7])
d_pos_w = np.linalg.norm(anchor_w - positive_w)
d_neg_w = np.linalg.norm(anchor_w - negative_w)
print("positive distance:", round(d_pos_w, 3))
print("negative distance:", round(d_neg_w, 3))
assert round(d_pos_w, 3) == 0.224
assert round(d_neg_w, 3) == 1.389

▶ What you'll see: the positive point is much closer to the anchor than the negative point.

In [ ]:
margin_w = 0.5
triplet_w = max(0.0, d_pos_w - d_neg_w + margin_w)
print("triplet loss:", round(triplet_w, 3))
assert round(triplet_w, 3) == 0.0

▶ What you'll see: the negative is already far enough away, so this triplet pays no loss.

In [ ]:
hard_negative_w = np.array([1.35, 1.05])
d_hard_w = np.linalg.norm(anchor_w - hard_negative_w)
triplet_hard_w = max(0.0, d_pos_w - d_hard_w + margin_w)
print("hard negative distance:", round(d_hard_w, 3))
print("hard triplet loss:", round(triplet_hard_w, 3))
assert round(triplet_hard_w, 3) == 0.370

▶ What you'll see: a nearby negative violates the margin and creates positive loss.

In [ ]:
plt.figure(figsize=(4.4, 3.4))
pts_w = np.vstack([anchor_w, positive_w, negative_w, hard_negative_w])
plt.scatter(pts_w[:, 0], pts_w[:, 1], s=[120, 90, 90, 90], color=["black", "seagreen", "gray", "crimson"])
for pt_w, name_w in zip(pts_w, ["anchor", "positive", "easy neg", "hard neg"]):
    plt.text(pt_w[0] + 0.03, pt_w[1] + 0.03, name_w)
plt.title("7: distance-based losses")
plt.xlabel("embedding dim 0")
plt.ylabel("embedding dim 1")
plt.show()

▶ What you'll see: the hard negative sits inside the desired separation zone; the easy negative does not.

*Why it's done this way:* these losses do not ask for a numeric target like 5 or a class probability like 0.9. They ask the geometry itself to become useful: similar examples should be close, and dissimilar examples should be separated by a margin.


## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per mechanic in this lesson. Each uses a handful of small
> numbers, prints every intermediate value with an inline `# ->` showing the result, draws one
> picture, and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · Residuals become scalar losses

Residuals keep direction, but losses turn them into nonnegative penalties that can be averaged.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t1_y = np.array([2.0, -1.0, 3.0, 0.0, 4.0, 1.0])  # -> [2.0, -1.0, 3.0, 0.0, 4.0, 1.0]
print("targets:", t1_y.tolist())  # -> [2.0, -1.0, 3.0, 0.0, 4.0, 1.0]
t1_yhat = np.array([1.5, -0.5, 4.0, -1.0, 3.5, 1.5])  # -> [1.5, -0.5, 4.0, -1.0, 3.5, 1.5]
print("predictions:", t1_yhat.tolist())  # -> [1.5, -0.5, 4.0, -1.0, 3.5, 1.5]
t1_resid = t1_yhat - t1_y  # -> [-0.5, 0.5, 1.0, -1.0, -0.5, 0.5]
print("residuals:", t1_resid.tolist())  # -> [-0.5, 0.5, 1.0, -1.0, -0.5, 0.5]
t1_abs = np.abs(t1_resid)  # -> [0.5, 0.5, 1.0, 1.0, 0.5, 0.5]
print("absolute penalties:", t1_abs.tolist())  # -> [0.5, 0.5, 1.0, 1.0, 0.5, 0.5]
t1_sq = t1_resid ** 2  # -> [0.25, 0.25, 1.0, 1.0, 0.25, 0.25]
print("squared penalties:", t1_sq.tolist())  # -> [0.25, 0.25, 1.0, 1.0, 0.25, 0.25]
t1_mae = np.mean(t1_abs)  # -> 0.6666666666666666
print("MAE:", round(float(t1_mae), 3))  # -> 0.667
t1_mse = np.mean(t1_sq)  # -> 0.5
print("MSE:", round(float(t1_mse), 3))  # -> 0.5
t1_index = np.arange(t1_y.size)  # -> [0, 1, 2, 3, 4, 5]
print("example index:", t1_index.tolist())  # -> [0, 1, 2, 3, 4, 5]
assert round(float(t1_mae), 3) == 0.667
assert round(float(t1_mse), 3) == 0.5

plt.figure(figsize=(4.8, 2.8))
plt.bar(t1_index - 0.18, t1_abs, width=0.36, label="absolute", color="gray")
plt.bar(t1_index + 0.18, t1_sq, width=0.36, label="squared", color="purple")
plt.xlabel("example")
plt.ylabel("penalty")
plt.title("Toy 1 · residual penalties")
plt.legend()
plt.show()

▶ What you'll see: the same six residuals make different scalar losses: MAE `0.667` and MSE `0.5`.

### ✍️ Toy 2 · MSE gradients scale with residual size

For mean squared error, the prediction gradient is `2 * residual / n`, so larger misses push harder.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t2_resid = np.array([-2.0, -1.0, -0.5, 0.0, 1.0, 2.0])  # -> [-2.0, -1.0, -0.5, 0.0, 1.0, 2.0]
print("residuals:", t2_resid.tolist())  # -> [-2.0, -1.0, -0.5, 0.0, 1.0, 2.0]
t2_penalty = t2_resid ** 2  # -> [4.0, 1.0, 0.25, 0.0, 1.0, 4.0]
print("squared penalties:", t2_penalty.tolist())  # -> [4.0, 1.0, 0.25, 0.0, 1.0, 4.0]
t2_mse = np.mean(t2_penalty)  # -> 1.7083333333333333
print("mean squared error:", round(float(t2_mse), 3))  # -> 1.708
t2_grad = 2 * t2_resid / t2_resid.size  # -> [-0.6666667, -0.3333333, -0.1666667, 0.0, 0.3333333, 0.6666667]
print("MSE gradient:", np.round(t2_grad, 3).tolist())  # -> [-0.667, -0.333, -0.167, 0.0, 0.333, 0.667]
t2_index = np.arange(t2_resid.size)  # -> [0, 1, 2, 3, 4, 5]
print("example index:", t2_index.tolist())  # -> [0, 1, 2, 3, 4, 5]
assert round(float(t2_mse), 3) == 1.708
assert np.allclose(np.round(t2_grad, 3), [-0.667, -0.333, -0.167, 0.0, 0.333, 0.667])

plt.figure(figsize=(4.8, 2.8))
plt.bar(t2_index, t2_penalty, color="lavender", label="squared loss")
plt.plot(t2_index, t2_grad, marker="o", color="crimson", label="gradient")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("example")
plt.title("Toy 2 · MSE loss and gradient")
plt.legend()
plt.show()

▶ What you'll see: the ±2 residuals have both the largest squared penalties and the largest gradient magnitudes.

### ✍️ Toy 3 · Cross-entropy explodes near zero probability

Cross-entropy is just `-log(p_true)`, so a tiny true-class probability is punished sharply.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t3_probs = np.array([0.70, 0.15, 0.10, 0.04, 0.01, 0.50])  # -> [0.7, 0.15, 0.1, 0.04, 0.01, 0.5]
print("true-class probabilities:", t3_probs.tolist())  # -> [0.7, 0.15, 0.1, 0.04, 0.01, 0.5]
t3_losses = -np.log(t3_probs)  # -> [0.3566749, 1.89712, 2.3025851, 3.2188758, 4.6051702, 0.6931472]
print("cross-entropies:", np.round(t3_losses, 3).tolist())  # -> [0.357, 1.897, 2.303, 3.219, 4.605, 0.693]
t3_ratio = t3_losses[4] / t3_losses[0]  # -> 12.911392471625762
print("loss ratio 0.01 vs 0.70:", round(float(t3_ratio), 3))  # -> 12.911
t3_index = np.arange(t3_probs.size)  # -> [0, 1, 2, 3, 4, 5]
print("example index:", t3_index.tolist())  # -> [0, 1, 2, 3, 4, 5]
assert round(float(t3_losses[4]), 3) == 4.605
assert round(float(t3_ratio), 3) == 12.911

plt.figure(figsize=(4.8, 2.8))
plt.plot(t3_probs, t3_losses, marker="o", color="crimson")
plt.xlabel("p(true class)")
plt.ylabel("-log p")
plt.title("Toy 3 · CE punishes low probability")
plt.show()

▶ What you'll see: moving from probability `0.70` to `0.01` makes the loss about `12.9×` larger.

### ✍️ Toy 4 · Stable softmax normalizes logits

Subtracting the maximum logit keeps exponentials safe while producing the same probabilities.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t4_logits = np.array([3.0, 1.0, 0.0, -1.0, 2.0, -2.0])  # -> [3.0, 1.0, 0.0, -1.0, 2.0, -2.0]
print("logits:", t4_logits.tolist())  # -> [3.0, 1.0, 0.0, -1.0, 2.0, -2.0]
t4_shifted = t4_logits - np.max(t4_logits)  # -> [0.0, -2.0, -3.0, -4.0, -1.0, -5.0]
print("shifted logits:", t4_shifted.tolist())  # -> [0.0, -2.0, -3.0, -4.0, -1.0, -5.0]
t4_exp = np.exp(t4_shifted)  # -> [1.0, 0.1353353, 0.0497871, 0.0183156, 0.3678794, 0.0067379]
print("exp shifted:", np.round(t4_exp, 3).tolist())  # -> [1.0, 0.135, 0.05, 0.018, 0.368, 0.007]
t4_denom = np.sum(t4_exp)  # -> 1.5780553786637386
print("normalizer:", round(float(t4_denom), 3))  # -> 1.578
t4_probs = t4_exp / t4_denom  # -> [0.6336913, 0.0857608, 0.0315496, 0.0116065, 0.233122, 0.0042698]
print("softmax probabilities:", np.round(t4_probs, 3).tolist())  # -> [0.634, 0.086, 0.032, 0.012, 0.233, 0.004]
t4_ce = -np.log(t4_probs[0])  # -> 0.4561933160181224
print("CE for class 0:", round(float(t4_ce), 3))  # -> 0.456
assert round(float(t4_probs[0]), 3) == 0.634
assert round(float(t4_ce), 3) == 0.456

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(t4_probs.size), t4_probs, color="teal")
plt.xlabel("class")
plt.ylabel("probability")
plt.title("Toy 4 · stable softmax")
plt.show()

▶ What you'll see: the largest logit gets probability `0.634`, and all six probabilities sum to one.

### ✍️ Toy 5 · Softmax-CE gradient is p minus y

For softmax with cross-entropy, one vector subtraction gives the logit gradient.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t5_logits = np.array([0.0, 1.0, 2.0, -1.0, 0.5, -0.5])  # -> [0.0, 1.0, 2.0, -1.0, 0.5, -0.5]
print("logits:", t5_logits.tolist())  # -> [0.0, 1.0, 2.0, -1.0, 0.5, -0.5]
t5_shifted = t5_logits - np.max(t5_logits)  # -> [-2.0, -1.0, 0.0, -3.0, -1.5, -2.5]
print("shifted logits:", t5_shifted.tolist())  # -> [-2.0, -1.0, 0.0, -3.0, -1.5, -2.5]
t5_exp = np.exp(t5_shifted)  # -> [0.1353353, 0.3678794, 1.0, 0.0497871, 0.2231302, 0.082085]
print("exp shifted:", np.round(t5_exp, 3).tolist())  # -> [0.135, 0.368, 1.0, 0.05, 0.223, 0.082]
t5_probs = t5_exp / np.sum(t5_exp)  # -> [0.072714, 0.197625, 0.53815, 0.026755, 0.119872, 0.044884]
print("probabilities:", np.round(t5_probs, 3).tolist())  # -> [0.073, 0.198, 0.538, 0.027, 0.12, 0.044]
t5_onehot = np.array([0.0, 0.0, 1.0, 0.0, 0.0, 0.0])  # -> [0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
print("one-hot target:", t5_onehot.tolist())  # -> [0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
t5_grad = t5_probs - t5_onehot  # -> [0.072714, 0.197625, -0.46185, 0.026755, 0.119872, 0.044884]
print("gradient p-y:", np.round(t5_grad, 3).tolist())  # -> [0.073, 0.198, -0.462, 0.027, 0.12, 0.044]
t5_eta = 0.5  # -> 0.5
print("learning rate:", t5_eta)  # -> 0.5
t5_step = t5_eta * t5_grad  # -> [0.036357, 0.0988125, -0.230925, 0.0133775, 0.059936, 0.022442]
print("logit step:", np.round(t5_step, 3).tolist())  # -> [0.036, 0.099, -0.231, 0.013, 0.06, 0.022]
t5_new_logits = t5_logits - t5_step  # -> [-0.036357, 0.9011875, 2.230925, -1.0133775, 0.440064, -0.522442]
print("new logits:", np.round(t5_new_logits, 3).tolist())  # -> [-0.036, 0.901, 2.231, -1.013, 0.44, -0.522]
t5_new_exp = np.exp(t5_new_logits - np.max(t5_new_logits))  # -> [0.103539, 0.264293, 1.0, 0.039, 0.166]
print("new exp shifted:", np.round(t5_new_exp, 3).tolist())  # -> [0.104, 0.264, 1.0, 0.039, 0.167, 0.063]
t5_new_probs = t5_new_exp / np.sum(t5_new_exp)  # -> [0.063281, 0.161509, 0.610643, 0.023818, 0.101858, 0.038891]
print("new probabilities:", np.round(t5_new_probs, 3).tolist())  # -> [0.063, 0.162, 0.611, 0.024, 0.102, 0.039]
assert round(float(np.sum(t5_grad)), 6) == 0.0
assert t5_new_probs[2] > t5_probs[2]

plt.figure(figsize=(4.4, 2.8))
plt.bar(["old p_true", "new p_true"], [t5_probs[2], t5_new_probs[2]], color=["gray", "seagreen"])
plt.ylim(0, 1)
plt.title("Toy 5 · descent raises true probability")
plt.show()

▶ What you'll see: subtracting `p-y` raises the true-class probability from `0.538` to `0.611`.

### ✍️ Toy 6 · Hinge loss pushes margin violators

Hinge loss gives a constant push until the signed margin reaches one, then goes flat.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t6_scores = np.array([-1.0, -0.2, 0.4, 0.9, 1.1, 2.0])  # -> [-1.0, -0.2, 0.4, 0.9, 1.1, 2.0]
print("scores:", t6_scores.tolist())  # -> [-1.0, -0.2, 0.4, 0.9, 1.1, 2.0]
t6_labels = np.ones_like(t6_scores)  # -> [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
print("labels:", t6_labels.tolist())  # -> [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
t6_margins = t6_labels * t6_scores  # -> [-1.0, -0.2, 0.4, 0.9, 1.1, 2.0]
print("signed margins:", t6_margins.tolist())  # -> [-1.0, -0.2, 0.4, 0.9, 1.1, 2.0]
t6_hinge = np.maximum(0.0, 1.0 - t6_margins)  # -> [2.0, 1.2, 0.6, 0.1, 0.0, 0.0]
print("hinge losses:", t6_hinge.tolist())  # -> [2.0, 1.2, 0.6, 0.1, 0.0, 0.0]
t6_grad = np.where(t6_margins < 1.0, -t6_labels, 0.0)  # -> [-1.0, -1.0, -1.0, -1.0, 0.0, 0.0]
print("hinge gradients:", t6_grad.tolist())  # -> [-1.0, -1.0, -1.0, -1.0, 0.0, 0.0]
t6_mean = np.mean(t6_hinge)  # -> 0.65
print("mean hinge loss:", round(float(t6_mean), 3))  # -> 0.65
assert np.allclose(t6_hinge, [2.0, 1.2, 0.6, 0.1, 0.0, 0.0])
assert np.allclose(t6_grad, [-1.0, -1.0, -1.0, -1.0, 0.0, 0.0])

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(t6_scores.size), t6_hinge, color="darkorange")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("example")
plt.ylabel("hinge loss")
plt.title("Toy 6 · zero after margin 1")
plt.show()

▶ What you'll see: only examples with margin below `1` pay loss and receive gradient.

### ✍️ Toy 7 · Triplet loss compares distances

A triplet loss is positive only when the negative is not far enough beyond the positive.

In [ ]:
import numpy as np

t7_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t7_anchor = np.array([0.0, 0.0])  # -> [0.0, 0.0]
print("anchor:", t7_anchor.tolist())  # -> [0.0, 0.0]
t7_positive = np.array([1.0, 0.0])  # -> [1.0, 0.0]
print("positive:", t7_positive.tolist())  # -> [1.0, 0.0]
t7_easy_negative = np.array([3.0, 0.0])  # -> [3.0, 0.0]
print("easy negative:", t7_easy_negative.tolist())  # -> [3.0, 0.0]
t7_hard_negative = np.array([1.2, 0.4])  # -> [1.2, 0.4]
print("hard negative:", t7_hard_negative.tolist())  # -> [1.2, 0.4]
t7_d_pos = np.linalg.norm(t7_anchor - t7_positive)  # -> 1.0
print("positive distance:", round(float(t7_d_pos), 3))  # -> 1.0
t7_d_easy = np.linalg.norm(t7_anchor - t7_easy_negative)  # -> 3.0
print("easy negative distance:", round(float(t7_d_easy), 3))  # -> 3.0
t7_d_hard = np.linalg.norm(t7_anchor - t7_hard_negative)  # -> 1.2649110640673518
print("hard negative distance:", round(float(t7_d_hard), 3))  # -> 1.265
t7_margin = 0.75  # -> 0.75
print("margin:", t7_margin)  # -> 0.75
t7_easy_loss = max(0.0, t7_d_pos - t7_d_easy + t7_margin)  # -> 0.0
print("easy triplet loss:", round(float(t7_easy_loss), 3))  # -> 0.0
t7_hard_loss = max(0.0, t7_d_pos - t7_d_hard + t7_margin)  # -> 0.48508893593264824
print("hard triplet loss:", round(float(t7_hard_loss), 3))  # -> 0.485
assert round(float(t7_easy_loss), 3) == 0.0
assert round(float(t7_hard_loss), 3) == 0.485

t7_points = np.vstack([t7_anchor, t7_positive, t7_easy_negative, t7_hard_negative])  # -> [[0,0],[1,0],[3,0],[1.2,0.4]]
print("points:\n", np.round(t7_points, 3))  # -> [[0.  0. ], [1.  0. ], [3.  0. ], [1.2 0.4]]
plt.figure(figsize=(4.6, 3.2))
plt.scatter(t7_points[:, 0], t7_points[:, 1], s=[120, 90, 90, 90], color=["black", "seagreen", "gray", "crimson"])
plt.text(0.05, 0.05, "anchor")
plt.text(1.05, 0.05, "positive")
plt.text(3.05, 0.05, "easy neg")
plt.text(1.25, 0.45, "hard neg")
plt.title("Toy 7 · hard negative violates margin")
plt.xlabel("embedding dim 0")
plt.ylabel("embedding dim 1")
plt.show()

▶ What you'll see: the hard negative is close enough to create triplet loss `0.485`; the easy negative pays zero.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, vectorized loss formulas, gradients, and assertions.
import matplotlib.pyplot as plt # load Matplotlib for compact diagnostic plots.
np.random.seed(0) # make all examples reproducible across runs.

def softmax(z): # convert logits into probabilities in a numerically stable way.
    z = np.asarray(z, dtype=float) # ensure vectorized float arithmetic.
    shifted = z - np.max(z) # subtract the largest logit to keep exponentials finite.
    exp_z = np.exp(shifted) # exponentiate shifted logits.
    return exp_z / np.sum(exp_z) # normalize into probabilities that sum to one.

def mse(yhat, y): # compute mean squared error for regression vectors.
    yhat = np.asarray(yhat, dtype=float) # convert predictions to floats.
    y = np.asarray(y, dtype=float) # convert targets to floats.
    return float(np.mean((yhat - y) ** 2)) # average squared residuals.

def cross_entropy_from_probs(p, true_idx): # compute CE when probabilities are already available.
    p = np.asarray(p, dtype=float) # convert probabilities to floats.
    return float(-np.log(p[int(true_idx)] + 1e-12)) # add epsilon so log never sees exact zero.

def show_loss_curve(x, y, title, xlabel, ylabel): # small plotting helper for loss curves.
    plt.figure(figsize=(4, 3)) # create a compact figure.
    plt.plot(x, y, color="purple") # draw the curve.
    plt.title(title) # title the plot.
    plt.xlabel(xlabel) # label the x-axis.
    plt.ylabel(ylabel) # label the y-axis.
    plt.show() # display the figure.

## 🟢 Basics (warm-up)

### Basic 1 — Compute residuals

**Goal.** Start with signed prediction errors, because every regression loss begins by measuring how far predictions are from targets. We build it in 2 steps.

In [ ]:
y_b1 = np.array([3.0, 1.0, -2.0]) # define three regression targets.
yhat_b1 = np.array([2.5, 1.5, -1.0]) # define three predictions to compare with the targets.
resid_b1 = yhat_b1 - y_b1 # compute signed residuals as prediction minus target.
print("residuals:", resid_b1) # inspect the direction and size of each mistake.
assert np.allclose(resid_b1, [-0.5, 0.5, 1.0]) # verify the worked residuals.

▶ What you'll see: one underprediction, one overprediction, and one larger overprediction.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact residual plot.
plt.bar(["case 0", "case 1", "case 2"], resid_b1, color="steelblue") # draw one bar per signed error.
plt.axhline(0, color="black", linewidth=0.8) # add a zero line for direction.
plt.title("Basic 1: signed residuals") # title the plot.
plt.ylabel("prediction - target") # label residual units.
plt.show() # display the plot.

▶ What you'll see: bars below zero are underpredictions; bars above zero are overpredictions.

👀 Takeaway: residuals carry both size and direction, while a loss decides how to penalize them.

### Basic 2 — Mean squared error

**Goal.** Turn residuals into MSE, because squared error is the standard smooth regression objective. We build it in 2 steps.

In [ ]:
y_b2 = np.array([3.0, 1.0, -2.0]) # define targets for the MSE example.
yhat_b2 = np.array([2.5, 1.5, -1.0]) # define predictions with known residuals.
squared_b2 = (yhat_b2 - y_b2) ** 2 # square each residual to remove sign and emphasize large misses.
print("squared errors:", squared_b2) # inspect each per-example penalty.
assert np.allclose(squared_b2, [0.25, 0.25, 1.0]) # verify the squared errors.

▶ What you'll see: the largest residual contributes four times as much as each half-unit residual.

In [ ]:
mse_b2 = np.mean(squared_b2) # average the per-example squared errors.
print("MSE:", round(mse_b2, 3)) # inspect the scalar objective.
assert round(mse_b2, 3) == 0.500 # verify the worked MSE.
plt.figure(figsize=(4, 3)) # create a compact penalty plot.
plt.bar(["case 0", "case 1", "case 2"], squared_b2, color="purple") # show squared penalties.
plt.title("Basic 2: squared errors") # title the plot.
plt.ylabel("(ŷ-y)^2") # label the penalty scale.
plt.show() # display the plot.

▶ What you'll see: the largest mistake dominates the average more than it did as a raw residual.

👀 Takeaway: MSE is smooth and simple, but it deliberately amplifies large errors.

### Basic 3 — MSE gradient

**Goal.** Compute the gradient of MSE with respect to predictions, because optimization updates predictions through this signal. We build it in 2 steps.

In [ ]:
y_b3 = np.array([3.0, 1.0, -2.0]) # define targets.
yhat_b3 = np.array([2.5, 1.5, -1.0]) # define predictions.
grad_b3 = 2 * (yhat_b3 - y_b3) / len(y_b3) # compute d/dŷ mean((ŷ-y)^2).
print("MSE gradient:", np.round(grad_b3, 3)) # inspect the training signal per prediction.
assert np.allclose(np.round(grad_b3, 3), [-0.333, 0.333, 0.667]) # verify the gradient values.

▶ What you'll see: gradient signs match whether each prediction needs to rise or fall under gradient descent.

In [ ]:
eta_b3 = 0.3 # choose a small learning rate for one prediction-space update.
yhat_new_b3 = yhat_b3 - eta_b3 * grad_b3 # move predictions opposite the gradient.
print("old MSE:", round(mse(yhat_b3, y_b3), 3), "new MSE:", round(mse(yhat_new_b3, y_b3), 3)) # compare loss before and after.
assert mse(yhat_new_b3, y_b3) < mse(yhat_b3, y_b3) # verify the update reduced MSE.

▶ What you'll see: one gradient step lowers the MSE.

In [ ]:
pred_grid_b3 = np.linspace(-0.5, 3.5, 200) # sweep one prediction while holding the others fixed.
loss_curve_b3 = np.array([mse(np.array([p, yhat_b3[1], yhat_b3[2]]), y_b3) for p in pred_grid_b3]) # compute MSE curve for case 0.
tangent_b3 = mse(yhat_b3, y_b3) + grad_b3[0] * (pred_grid_b3 - yhat_b3[0]) # draw the local gradient slope.
plt.figure(figsize=(4, 3)) # create gradient visualization.
plt.plot(pred_grid_b3, loss_curve_b3, label="MSE vs ŷ₀", color="purple") # show the loss curve.
plt.plot(pred_grid_b3, tangent_b3, linestyle="--", label="gradient slope", color="gray") # show local slope.
plt.scatter([yhat_b3[0], y_b3[0]], [mse(yhat_b3, y_b3), mse(np.array([y_b3[0], yhat_b3[1], yhat_b3[2]]), y_b3)], color=["crimson", "seagreen"]) # mark current and target predictions.
plt.title("Basic 3: MSE gradient slope") # title the plot.
plt.xlabel("first prediction ŷ₀") # label x-axis.
plt.ylabel("MSE") # label y-axis.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: the dashed tangent slopes downward toward the target for the under-predicted first case.

👀 Takeaway: the MSE gradient points away from the target, so subtracting it moves predictions toward the target.

### Basic 4 — Mean absolute error

**Goal.** Compare MAE with MSE, because absolute error penalizes each unit of residual linearly. We build it in 2 steps.

In [ ]:
resid_b4 = np.array([-0.5, 0.5, 1.0]) # reuse a small residual vector.
abs_errors_b4 = np.abs(resid_b4) # compute absolute penalties.
print("absolute errors:", abs_errors_b4) # inspect per-example MAE terms.
assert np.allclose(abs_errors_b4, [0.5, 0.5, 1.0]) # verify absolute errors.

▶ What you'll see: signs disappear, but a 1.0 residual is only twice a 0.5 residual.

In [ ]:
mae_b4 = np.mean(abs_errors_b4) # average absolute errors.
print("MAE:", round(mae_b4, 3)) # inspect the scalar objective.
assert round(mae_b4, 3) == 0.667 # verify the MAE.
plt.figure(figsize=(4, 3)) # create a comparison chart.
plt.bar(["MAE", "MSE"], [mae_b4, np.mean(resid_b4 ** 2)], color=["gray", "purple"]) # compare scalar losses.
plt.title("Basic 4: MAE vs MSE") # title the plot.
plt.ylabel("loss") # label the loss scale.
plt.show() # display the comparison.

▶ What you'll see: MAE and MSE summarize the same residuals differently.

👀 Takeaway: MAE is less aggressive toward large residuals than MSE.

### Basic 5 — Binary cross-entropy

**Goal.** Compute binary cross-entropy, because two-class probability models must penalize confident wrong probabilities. We build it in 2 steps.

In [ ]:
y_b5 = np.array([1.0, 0.0, 1.0]) # define binary labels.
p_b5 = np.array([0.8, 0.3, 0.1]) # define predicted probabilities for class 1.
terms_b5 = -(y_b5 * np.log(p_b5) + (1 - y_b5) * np.log(1 - p_b5)) # compute BCE term by term.
print("BCE terms:", np.round(terms_b5, 3)) # inspect the per-example penalties.
assert np.allclose(np.round(terms_b5, 3), [0.223, 0.357, 2.303]) # verify the terms.

▶ What you'll see: predicting 0.1 for a positive label is the costly term.

In [ ]:
bce_b5 = float(np.mean(terms_b5)) # average binary cross-entropy terms.
print("mean BCE:", round(bce_b5, 3)) # inspect the scalar objective.
assert round(bce_b5, 3) == 0.961 # verify the worked BCE.
plt.figure(figsize=(4, 3)) # create a per-example penalty plot.
plt.bar(["ex0", "ex1", "ex2"], terms_b5, color="crimson") # show BCE penalties.
plt.title("Basic 5: binary cross-entropy terms") # title the plot.
plt.ylabel("loss") # label the loss axis.
plt.show() # display the plot.

▶ What you'll see: the confidently wrong positive example dominates the average.

👀 Takeaway: cross-entropy is small for confident correct probabilities and large for confident wrong ones.

### Basic 6 — Stable softmax

**Goal.** Convert logits into probabilities safely, because cross-entropy expects probabilities but models produce raw scores. We build it in 2 steps.

In [ ]:
logits_b6 = np.array([2.15, 0.40]) # define the two scores from the lesson arithmetic.
exp_b6 = np.exp(logits_b6 - np.max(logits_b6)) # exponentiate shifted logits for numerical stability.
probs_b6 = exp_b6 / np.sum(exp_b6) # normalize shifted exponentials into probabilities.
print("softmax probabilities:", np.round(probs_b6, 3)) # inspect the calibrated comparison.
assert round(probs_b6[0], 3) == 0.852 # verify the lesson's two-class probability.

▶ What you'll see: score 2.15 beats baseline 0.40 with probability about 0.852.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact probability plot.
plt.bar(["score 2.15", "score 0.40"], probs_b6, color="teal") # show the softmax comparison.
plt.title("Basic 6: two-class softmax") # title the plot.
plt.ylabel("probability") # label probability scale.
plt.ylim(0, 1) # keep probability axis bounded.
plt.show() # display the plot.

▶ What you'll see: the larger logit receives most, but not all, of the probability mass.

👀 Takeaway: softmax turns relative score gaps into normalized probabilities.

### Basic 7 — Multiclass cross-entropy

**Goal.** Compute multiclass CE from softmax probabilities, because classification training usually combines these two steps. We build it in 2 steps.

In [ ]:
logits_b7 = np.array([1.0, 2.0, 0.0]) # define three class logits.
true_b7 = 1 # set the true class to index 1.
probs_b7 = softmax(logits_b7) # convert logits to probabilities.
print("probabilities:", np.round(probs_b7, 3)) # inspect the distribution over classes.
assert round(probs_b7[true_b7], 3) == 0.665 # verify true-class probability.

▶ What you'll see: class 1 has the highest probability because its logit is largest.

In [ ]:
ce_b7 = cross_entropy_from_probs(probs_b7, true_b7) # compute -log probability of the true class.
print("cross-entropy:", round(ce_b7, 3)) # inspect the scalar loss.
assert round(ce_b7, 3) == 0.408 # verify CE from p_true ≈ 0.665.
plt.figure(figsize=(4, 3)) # create a probability chart.
plt.bar(["class 0", "class 1", "class 2"], probs_b7, color="darkcyan") # show class probabilities.
plt.title("Basic 7: CE reads the true-class bar") # title the plot.
plt.ylabel("probability") # label probability scale.
plt.show() # display the chart.

▶ What you'll see: CE uses only the probability assigned to the true class after normalization.

👀 Takeaway: multiclass CE is low exactly when the true class receives high probability.

### Basic 8 — CE gradient as p minus y

**Goal.** Compute the softmax-cross-entropy gradient, because this compact signal drives classifier training. We build it in 2 steps.

In [ ]:
probs_b8 = softmax(np.array([1.0, 2.0, 0.0])) # compute probabilities for a three-class example.
y_b8 = np.array([0.0, 1.0, 0.0]) # one-hot target for class 1.
grad_b8 = probs_b8 - y_b8 # compute dL/dlogits for softmax plus CE.
print("gradient:", np.round(grad_b8, 3)) # inspect which logits rise or fall under descent.
assert round(float(np.sum(grad_b8)), 6) == 0.0 # verify probability mass is redistributed.

▶ What you'll see: the true class has a negative gradient; the wrong classes have positive gradients.

In [ ]:
logits_b8 = np.array([1.0, 2.0, 0.0]) # define original logits.
new_logits_b8 = logits_b8 - 0.5 * grad_b8 # take one gradient descent step.
new_probs_b8 = softmax(new_logits_b8) # convert updated logits to probabilities.
print("old true p:", round(probs_b8[1], 3), "new true p:", round(new_probs_b8[1], 3)) # compare true-class probability.
assert new_probs_b8[1] > probs_b8[1] # verify descent improved the true class.

▶ What you'll see: the true-class probability increases after one descent step.

In [ ]:
idx_b8 = np.arange(len(grad_b8)) # create class positions.
plt.figure(figsize=(4, 3)) # create gradient bar plot.
plt.axhline(0.0, color="black", linewidth=0.8) # mark zero gradient.
plt.bar(idx_b8, grad_b8, color=["crimson" if g > 0 else "seagreen" for g in grad_b8]) # show p-y per class.
plt.xticks(idx_b8, ["class 0", "class 1", "class 2"]) # label classes.
plt.title("Basic 8: softmax-CE gradient") # title the plot.
plt.ylabel("p - y") # label gradient axis.
plt.show() # display the plot.

▶ What you'll see: wrong classes have positive bars to push logits down, while the true class has a negative bar to push its logit up.

👀 Takeaway: $p-y$ is a probability redistribution instruction.

### Basic 9 — Hinge margin loss

**Goal.** Compute hinge loss for signed labels, because margin losses train classifiers to be correct by a safe gap. We build it in 2 steps.

In [ ]:
scores_b9 = np.array([1.5, 0.2, -0.5]) # define classifier scores.
labels_b9 = np.array([1.0, 1.0, -1.0]) # define signed labels in {-1,+1}.
margin_b9 = labels_b9 * scores_b9 # compute signed margins y*s.
losses_b9 = np.maximum(0.0, 1 - margin_b9) # compute hinge penalties.
print("margins:", margin_b9) # inspect correctness with margin.
print("hinge losses:", losses_b9) # inspect penalties.
assert np.allclose(losses_b9, [0.0, 0.8, 0.5]) # verify hinge losses.

▶ What you'll see: only examples with margin below 1 pay loss.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact margin plot.
plt.bar(["ex0", "ex1", "ex2"], losses_b9, color="darkorange") # show hinge penalties.
plt.title("Basic 9: hinge loss by example") # title the plot.
plt.ylabel("max(0, 1-y score)") # label loss formula.
plt.show() # display the plot.

▶ What you'll see: the already-safe example has zero loss.

👀 Takeaway: hinge loss trains until the margin is satisfied, then stops caring about extra confidence.

### Basic 10 — One scalar update from a gradient

**Goal.** Apply the lesson's update rule, because losses matter only because their gradients move parameters. We build it in 2 steps.

In [ ]:
theta_b10 = 2.0 # define one scalar parameter.
grad_b10 = 2.1 # define the scalar gradient from the loss.
eta_b10 = 0.06 # define the learning rate from the lesson arithmetic.
new_theta_b10 = theta_b10 - eta_b10 * grad_b10 # perform one gradient descent update.
print("updated parameter:", round(new_theta_b10, 3)) # inspect the new value.
assert round(new_theta_b10, 3) == 1.874 # verify 2.000 - 0.060*2.100.

▶ What you'll see: the parameter moves by 0.126, a small reliable nudge.

In [ ]:
plt.figure(figsize=(4, 3)) # create a before-after plot.
plt.bar(["before", "after"], [theta_b10, new_theta_b10], color=["gray", "seagreen"]) # compare parameter values.
plt.title("Basic 10: one gradient step") # title the plot.
plt.ylabel("parameter value") # label the scale.
plt.show() # display the plot.

▶ What you'll see: the parameter moves opposite the positive gradient.

👀 Takeaway: a loss becomes learning only after its gradient is scaled by the learning rate and applied to parameters.

## 🟡 Easy

### Easy 1 — Fit a line with MSE steps

**Goal.** Train one scalar weight with MSE, because regression loss directly creates a gradient for parameters. We build it in 3 steps.

In [ ]:
x_e1 = np.array([1.0, 2.0, 3.0]) # define one-dimensional inputs.
y_e1 = np.array([2.0, 4.0, 6.0]) # define targets from y=2x.
w_e1 = 0.0 # initialize the model weight.
print("initial weight:", w_e1) # inspect the starting parameter.

▶ What you'll see: the line starts flat because the weight is zero.

In [ ]:
losses_e1 = [] # store MSE values during training.
for step_e1 in range(25): # run a small batch-gradient loop.
    pred_e1 = w_e1 * x_e1 # compute predictions from the current weight.
    loss_e1 = mse(pred_e1, y_e1) # compute MSE.
    grad_e1 = 2 * np.mean((pred_e1 - y_e1) * x_e1) # compute dMSE/dw by the chain rule.
    w_e1 = w_e1 - 0.05 * grad_e1 # update the weight by gradient descent.
    losses_e1.append(loss_e1) # record the loss.
print("final weight:", round(w_e1, 3), "final loss:", round(losses_e1[-1], 4)) # inspect training result.
assert abs(w_e1 - 2.0) < 0.01 # verify the learned weight is near the true slope.

▶ What you'll see: the weight approaches 2 and the loss becomes tiny.

In [ ]:
plt.figure(figsize=(4, 3)) # create a learning-curve plot.
plt.plot(losses_e1, color="purple") # draw MSE over gradient steps.
plt.title("Easy 1: MSE training curve") # title the plot.
plt.xlabel("step") # label optimization steps.
plt.ylabel("MSE") # label loss scale.
plt.show() # display the plot.

▶ What you'll see: loss drops quickly as the slope moves toward the target relationship.

👀 Takeaway: MSE gradients combine residual size with input size to move regression parameters.

### Easy 2 — Train a logistic classifier with BCE

**Goal.** Use binary cross-entropy to separate two points, because probabilistic classification learns from log-loss gradients. We build it in 3 steps.

In [ ]:
x_e2 = np.array([-1.0, 1.0]) # define two one-dimensional examples.
y_e2 = np.array([0.0, 1.0]) # define binary labels.
w_e2 = 0.0 # initialize a scalar logit weight.
print("examples:", x_e2, "labels:", y_e2) # inspect the tiny dataset.

▶ What you'll see: the negative input belongs to class 0 and the positive input belongs to class 1.

In [ ]:
losses_e2 = [] # store BCE values.
for step_e2 in range(40): # run deterministic full-batch gradient descent.
    logits_e2 = w_e2 * x_e2 # compute logits.
    probs_e2 = 1 / (1 + np.exp(-logits_e2)) # apply sigmoid from scratch.
    loss_e2 = -np.mean(y_e2 * np.log(probs_e2 + 1e-12) + (1 - y_e2) * np.log(1 - probs_e2 + 1e-12)) # compute BCE.
    grad_e2 = np.mean((probs_e2 - y_e2) * x_e2) # BCE-with-sigmoid gradient wrt w.
    w_e2 = w_e2 - 0.4 * grad_e2 # update the weight.
    losses_e2.append(loss_e2) # store loss.
print("trained weight:", round(w_e2, 3), "last loss:", round(losses_e2[-1], 3)) # inspect result.
assert w_e2 > 2.0 # verify the model learned a positive separating weight.

▶ What you'll see: the positive weight makes x=1 likely class 1 and x=-1 likely class 0.

In [ ]:
plt.figure(figsize=(4, 3)) # create a BCE learning curve.
plt.plot(losses_e2, color="crimson") # draw loss over steps.
plt.title("Easy 2: BCE training curve") # title the curve.
plt.xlabel("step") # label steps.
plt.ylabel("binary cross-entropy") # label the loss.
plt.show() # display the curve.

▶ What you'll see: cross-entropy decreases as probabilities align with labels.

👀 Takeaway: BCE creates strong gradients when the probability assigned to the true label is too low.

### Easy 3 — Compare MSE and CE on probabilities

**Goal.** Evaluate the same classification probabilities with two losses, because MSE and CE express different ideas of wrongness. We build it in 3 steps.

In [ ]:
p_e3 = np.array([0.9, 0.6, 0.1]) # predicted probabilities for class 1.
y_e3 = np.array([1.0, 1.0, 1.0]) # all three examples are actually positive.
mse_terms_e3 = (p_e3 - y_e3) ** 2 # compute squared probability errors.
ce_terms_e3 = -np.log(p_e3) # compute positive-label cross-entropy terms.
print("MSE terms:", np.round(mse_terms_e3, 3)) # inspect MSE penalties.
print("CE terms:", np.round(ce_terms_e3, 3)) # inspect CE penalties.
assert np.allclose(np.round(ce_terms_e3, 3), [0.105, 0.511, 2.303]) # verify CE numbers.

▶ What you'll see: CE sharply penalizes the probability 0.1 on a positive label.

In [ ]:
mean_mse_e3 = float(np.mean(mse_terms_e3)) # average MSE terms.
mean_ce_e3 = float(np.mean(ce_terms_e3)) # average CE terms.
print("mean MSE:", round(mean_mse_e3, 3), "mean CE:", round(mean_ce_e3, 3)) # compare scalar losses.
assert round(mean_mse_e3, 3) == 0.327 # verify mean probability MSE.

▶ What you'll see: the two objectives are on different scales and emphasize different regions.

In [ ]:
plt.figure(figsize=(4, 3)) # create a grouped comparison plot.
idx_e3 = np.arange(3) # x positions for examples.
plt.bar(idx_e3 - 0.18, mse_terms_e3, width=0.36, label="MSE", color="gray") # plot MSE terms.
plt.bar(idx_e3 + 0.18, ce_terms_e3, width=0.36, label="CE", color="crimson") # plot CE terms.
plt.xticks(idx_e3, ["p=.9", "p=.6", "p=.1"]) # label examples by probability.
plt.title("Easy 3: probability losses") # title the comparison.
plt.legend() # show loss labels.
plt.show() # display the plot.

▶ What you'll see: CE grows much more dramatically as true-label probability approaches zero.

👀 Takeaway: CE is usually the better classification loss because it treats probabilities as beliefs.

### Easy 4 — Sweep a margin for triplet loss

**Goal.** Show how triplet margin changes loss, because embedding objectives define success through relative distances. We build it in 3 steps.

In [ ]:
d_pos_e4 = 0.4 # define anchor-positive distance.
d_neg_e4 = 0.8 # define anchor-negative distance.
margins_e4 = np.array([0.1, 0.3, 0.5, 0.7]) # choose margin values to sweep.
losses_e4 = np.maximum(0.0, d_pos_e4 - d_neg_e4 + margins_e4) # compute triplet losses.
print("triplet losses:", losses_e4) # inspect how the margin activates loss.
assert np.allclose(losses_e4, [0.0, 0.0, 0.1, 0.3]) # verify the sweep.

▶ What you'll see: small margins are already satisfied, but larger margins create loss.

In [ ]:
plt.figure(figsize=(4, 3)) # create a margin sweep plot.
plt.plot(margins_e4, losses_e4, marker="o", color="darkorange") # draw triplet loss by margin.
plt.title("Easy 4: triplet margin sweep") # title the plot.
plt.xlabel("margin") # label margin axis.
plt.ylabel("loss") # label loss axis.
plt.show() # display the plot.

▶ What you'll see: loss begins only when the requested margin exceeds the current distance gap.

In [ ]:
required_gap_e4 = margins_e4[-1] # choose the strictest margin.
actual_gap_e4 = d_neg_e4 - d_pos_e4 # compute current separation.
print("actual gap:", round(actual_gap_e4, 3), "required gap:", round(required_gap_e4, 3)) # inspect why the strict margin fails.
assert round(actual_gap_e4, 3) == 0.400 # verify current gap.

▶ What you'll see: the strictest margin asks for 0.7 separation, but the embedding has only 0.4.

👀 Takeaway: margin losses are constraint-like objectives: zero if the geometry is good enough, positive otherwise.

### Easy 5 — Add L2 regularization to a loss

**Goal.** Combine data loss and weight penalty, because loss functions often include constraints that control capacity. We build it in 3 steps.

In [ ]:
w_e5 = np.array([2.0, -1.0, 0.5]) # define model weights.
data_loss_e5 = 0.8 # define a data-fitting loss.
lam_e5 = 0.1 # define regularization strength.
penalty_e5 = lam_e5 * np.sum(w_e5 ** 2) # compute λ||w||².
print("L2 penalty:", round(penalty_e5, 3)) # inspect the added cost.
assert round(penalty_e5, 3) == 0.525 # verify penalty.

▶ What you'll see: large weights add cost even before considering data error.

In [ ]:
total_e5 = data_loss_e5 + penalty_e5 # combine data term and regularization term.
grad_reg_e5 = 2 * lam_e5 * w_e5 # compute gradient of λ||w||².
print("total loss:", round(total_e5, 3)) # inspect full objective.
print("regularization gradient:", grad_reg_e5) # inspect shrinkage direction.
assert round(total_e5, 3) == 1.325 # verify full loss.

▶ What you'll see: the regularizer pushes every weight back toward zero.

In [ ]:
plt.figure(figsize=(4, 3)) # create a loss breakdown plot.
plt.bar(["data", "L2 penalty", "total"], [data_loss_e5, penalty_e5, total_e5], color=["steelblue", "gray", "purple"]) # show objective pieces.
plt.title("Easy 5: regularized loss") # title the plot.
plt.ylabel("loss") # label loss scale.
plt.show() # display the breakdown.

▶ What you'll see: total loss is the data objective plus the capacity penalty.

👀 Takeaway: regularization changes the optimization target, not just the model after training.

## 🔴 Advanced

### Advanced 1 — Robust Huber loss

**Goal.** Build Huber loss, because it behaves like MSE near zero but like MAE for outliers. We build it in 3 steps.

In [ ]:
resid_a1 = np.array([-0.5, 0.2, 3.0]) # define residuals with one large outlier.
delta_a1 = 1.0 # set the quadratic-to-linear transition point.
abs_a1 = np.abs(resid_a1) # compute residual magnitudes.
huber_a1 = np.where(abs_a1 <= delta_a1, 0.5 * resid_a1 ** 2, delta_a1 * (abs_a1 - 0.5 * delta_a1)) # compute Huber terms.
print("Huber terms:", np.round(huber_a1, 3)) # inspect robust penalties.
assert np.allclose(np.round(huber_a1, 3), [0.125, 0.020, 2.500]) # verify Huber values.

▶ What you'll see: the outlier pays a linear penalty of 2.5 instead of squared penalty 9.

In [ ]:
mse_terms_a1 = resid_a1 ** 2 # compute squared penalties for comparison.
print("MSE terms:", mse_terms_a1) # inspect how much MSE charges the outlier.
assert mse_terms_a1[-1] == 9.0 # verify outlier squared penalty.

▶ What you'll see: MSE is far more dominated by the outlier.

In [ ]:
grid_a1 = np.linspace(-4, 4, 200) # define residual grid.
huber_curve_a1 = np.where(np.abs(grid_a1) <= delta_a1, 0.5 * grid_a1 ** 2, delta_a1 * (np.abs(grid_a1) - 0.5 * delta_a1)) # compute curve.
plt.figure(figsize=(4, 3)) # create robust loss plot.
plt.plot(grid_a1, huber_curve_a1, label="Huber", color="seagreen") # draw Huber.
plt.plot(grid_a1, 0.5 * grid_a1 ** 2, label="scaled MSE", color="gray", linestyle="--") # draw comparison.
plt.title("Advanced 1: Huber loss") # title the plot.
plt.xlabel("residual") # label x-axis.
plt.ylabel("loss") # label y-axis.
plt.legend() # show labels.
plt.show() # display curve.

▶ What you'll see: Huber matches a parabola near zero and becomes linear in the tails.

👀 Takeaway: robust losses reduce the influence of outliers without giving up smooth behavior near the optimum.

### Advanced 2 — Focal loss down-weights easy examples

**Goal.** Compute focal loss, because imbalanced classification often needs hard examples to count more than easy ones. We build it in 3 steps.

In [ ]:
p_true_a2 = np.array([0.95, 0.70, 0.20]) # predicted probabilities assigned to the true class.
gamma_a2 = 2.0 # set the focal focusing parameter.
ce_a2 = -np.log(p_true_a2) # compute standard cross-entropy terms.
focal_a2 = ((1 - p_true_a2) ** gamma_a2) * ce_a2 # compute focal loss terms.
print("CE terms:", np.round(ce_a2, 3)) # inspect standard CE.
print("focal terms:", np.round(focal_a2, 3)) # inspect down-weighted loss.
assert np.allclose(np.round(focal_a2, 3), [0.000, 0.032, 1.030]) # verify focal terms.

▶ What you'll see: the easy 0.95 example almost disappears, while the hard 0.20 example remains large.

In [ ]:
weights_a2 = (1 - p_true_a2) ** gamma_a2 # compute focal multipliers.
print("focal weights:", np.round(weights_a2, 3)) # inspect how easy examples are down-weighted.
assert np.allclose(np.round(weights_a2, 3), [0.003, 0.090, 0.640]) # verify weights.

▶ What you'll see: the weighting term is tiny for already-confident examples.

In [ ]:
plt.figure(figsize=(4, 3)) # create focal comparison plot.
idx_a2 = np.arange(3) # x positions.
plt.bar(idx_a2 - 0.18, ce_a2, width=0.36, label="CE", color="gray") # plot CE terms.
plt.bar(idx_a2 + 0.18, focal_a2, width=0.36, label="focal", color="crimson") # plot focal terms.
plt.xticks(idx_a2, ["p=.95", "p=.70", "p=.20"]) # label true-class probabilities.
plt.title("Advanced 2: focal loss") # title plot.
plt.legend() # show labels.
plt.show() # display comparison.

▶ What you'll see: focal loss concentrates more of the objective on hard examples.

👀 Takeaway: focal loss modifies cross-entropy so class imbalance does not let easy examples dominate training.

### Advanced 3 — Label smoothing changes targets

**Goal.** Apply label smoothing, because overconfident one-hot targets can make classifiers brittle. We build it in 3 steps.

In [ ]:
num_classes_a3 = 4 # define a four-class problem.
true_a3 = 2 # choose the true class index.
eps_a3 = 0.1 # choose smoothing strength.
y_smooth_a3 = np.full(num_classes_a3, eps_a3 / num_classes_a3) # give every class a small target mass.
y_smooth_a3[true_a3] += 1 - eps_a3 # put the remaining mass on the true class.
print("smoothed target:", y_smooth_a3) # inspect the target distribution.
assert np.allclose(y_smooth_a3, [0.025, 0.025, 0.925, 0.025]) # verify smoothed target.

▶ What you'll see: the true class is still dominant but no class has target exactly 0 or 1.

In [ ]:
probs_a3 = np.array([0.05, 0.05, 0.85, 0.05]) # define predicted probabilities.
ce_onehot_a3 = -np.log(probs_a3[true_a3]) # compute ordinary one-hot CE.
ce_smooth_a3 = -float(np.sum(y_smooth_a3 * np.log(probs_a3))) # compute smoothed CE.
print("one-hot CE:", round(ce_onehot_a3, 3), "smoothed CE:", round(ce_smooth_a3, 3)) # compare losses.
assert round(ce_onehot_a3, 3) == 0.163 # verify one-hot CE.

▶ What you'll see: smoothed CE also cares a little about non-true probabilities.

In [ ]:
plt.figure(figsize=(4, 3)) # create target distribution plot.
plt.bar(["c0", "c1", "c2", "c3"], y_smooth_a3, color="teal") # show smoothed labels.
plt.title("Advanced 3: label-smoothed target") # title the plot.
plt.ylabel("target probability") # label target mass.
plt.ylim(0, 1) # bound probability axis.
plt.show() # display target distribution.

▶ What you'll see: label smoothing softens the target while preserving the correct class.

👀 Takeaway: label smoothing changes the loss target so the model is discouraged from infinite confidence.

### Advanced 4 — Batch loss estimates are noisy

**Goal.** Compare minibatch losses with the full-data loss, because training usually optimizes noisy estimates of the objective. We build it in 3 steps.

In [ ]:
errors_a4 = np.array([0.1, -0.2, 0.3, 1.5, -1.2, 0.0, 0.4, -0.1]) # define residuals for eight examples.
full_mse_a4 = float(np.mean(errors_a4 ** 2)) # compute full-data MSE.
print("full MSE:", round(full_mse_a4, 3)) # inspect the complete objective.
assert round(full_mse_a4, 3) == 0.500 # verify full MSE.

▶ What you'll see: the full loss averages both small errors and two large ones.

In [ ]:
batch_ids_a4 = np.array([[0, 1], [2, 3], [4, 5], [6, 7]]) # define four minibatches of size 2.
batch_mse_a4 = np.array([np.mean(errors_a4[idx_a4] ** 2) for idx_a4 in batch_ids_a4]) # compute loss per minibatch.
print("batch MSEs:", np.round(batch_mse_a4, 3)) # inspect noisy estimates.
assert np.allclose(np.round(batch_mse_a4, 3), [0.025, 1.170, 0.720, 0.085]) # verify batch losses.

▶ What you'll see: minibatch losses vary widely depending on whether they include large residuals.

In [ ]:
plt.figure(figsize=(4, 3)) # create minibatch plot.
plt.bar(["b0", "b1", "b2", "b3"], batch_mse_a4, color="steelblue") # show batch losses.
plt.axhline(full_mse_a4, color="crimson", linestyle="--", label="full MSE") # mark full loss.
plt.title("Advanced 4: minibatch loss noise") # title the plot.
plt.ylabel("MSE estimate") # label loss estimate.
plt.legend() # show full loss reference.
plt.show() # display plot.

▶ What you'll see: each batch is an imperfect local estimate of the full objective.

👀 Takeaway: stochastic training follows noisy loss estimates, so trends matter more than a single batch value.

### Advanced 5 — Loss scale changes gradient scale

**Goal.** Show how multiplying a loss changes gradients, because scale affects stable learning rates and training dynamics. We build it in 3 steps.

In [ ]:
y_a5 = np.array([1.0, 2.0]) # define two targets.
yhat_a5 = np.array([0.0, 4.0]) # define predictions with residuals -1 and 2.
grad_mse_a5 = 2 * (yhat_a5 - y_a5) / len(y_a5) # compute MSE gradient wrt predictions.
scale_a5 = 10.0 # choose a loss multiplier.
grad_scaled_a5 = scale_a5 * grad_mse_a5 # compute gradient of 10*MSE.
print("MSE gradient:", grad_mse_a5) # inspect base gradient.
print("scaled gradient:", grad_scaled_a5) # inspect scaled gradient.
assert np.allclose(grad_mse_a5, [-1.0, 2.0]) # verify base gradient.

▶ What you'll see: multiplying the loss by 10 multiplies the gradient by 10.

In [ ]:
eta_a5 = 0.1 # choose a learning rate.
step_base_a5 = -eta_a5 * grad_mse_a5 # compute descent step for base loss.
step_scaled_a5 = -eta_a5 * grad_scaled_a5 # compute descent step for scaled loss.
print("base step:", step_base_a5) # inspect the normal update.
print("scaled step:", step_scaled_a5) # inspect the much larger update.
assert np.allclose(step_scaled_a5, 10 * step_base_a5) # verify scale relationship.

▶ What you'll see: the same learning rate becomes ten times more aggressive under the scaled loss.

In [ ]:
plt.figure(figsize=(4, 3)) # create gradient-scale comparison plot.
idx_a5 = np.arange(2) # x positions for predictions.
plt.bar(idx_a5 - 0.18, np.abs(step_base_a5), width=0.36, label="MSE step", color="gray") # plot base step magnitudes.
plt.bar(idx_a5 + 0.18, np.abs(step_scaled_a5), width=0.36, label="10× loss step", color="crimson") # plot scaled step magnitudes.
plt.xticks(idx_a5, ["ŷ0", "ŷ1"]) # label predictions.
plt.title("Advanced 5: loss scale controls step scale") # title the plot.
plt.ylabel("absolute update") # label update magnitude.
plt.legend() # show labels.
plt.show() # display comparison.

▶ What you'll see: scaled-loss updates are much larger, which may require a smaller learning rate.

👀 Takeaway: losses encode semantics and numerical scale; both matter for optimization stability.